In [11]:
from app import get_python_assistant
from dotenv import load_dotenv
import os
from fastapi import HTTPException
from pydantic import BaseModel

#### Documents

In [12]:
Documents = os.listdir("files")
Documents

['fastapi_tutorial.pdf',
 'Interview-level-QA-on-Python-Programming.pdf',
 'Introduction_to_Python_Programming.pdf',
 'learn-web-development-python-hands.pdf',
 'python_basics.pdf']

In [13]:
for d in Documents:
    print("documnets used:\n")
    print("-", d)

documnets used:

- fastapi_tutorial.pdf
documnets used:

- Interview-level-QA-on-Python-Programming.pdf
documnets used:

- Introduction_to_Python_Programming.pdf
documnets used:

- learn-web-development-python-hands.pdf
documnets used:

- python_basics.pdf


#### Initializing the assistant

In [14]:
assistant = get_python_assistant()

if not hasattr(assistant, 'builder'):
    raise ValueError("assistant.builder is not defined")

print("Assistant initialized")
print("Vector store documents:", assistant.vector_store._collection.count())

Assistant initialized
Vector store documents: 2850


#### Test Retrieval Tool

In [15]:
tool = assistant.tools[0]

query = "How does FastAPI dependency injection work?"
context = tool.invoke(query)

print("Retrieved context:\n")
print(context[:1500])


Retrieved context:

Source: fastapi_tutorial.pdf | Page: 80

22. FastAPI – DeFpasetAPnI d– Peytnhocni Weesb Framework
The built-in dependency injection system of FastAPI makes it possible to
integrate components easier when building your API. In programming,
Dependency injection refers to the mechanism where an object receives
other objects that it depends on. The other objects are called dependencies.
Dependency injection has the following advantages:
 reuse the same shared logic
 share database connections
 enforce authentication and security features
Assuming that a FastAPI app has two operation functions both having the
same query parameters id, name and age.
from fastapi import FastAPI
app = FastAPI()
@app.get("/user/")
async def user(id: str, name: str, age: int):
return {"id": id, "name": name, "age": age}
@app.get("/admin/")
async def admin(id: str, name: str, age: int):
return {"id": id, "name": name, "age": age}
In case of any changes such as adding/removing query paramete

In [16]:
load_dotenv()





def ask_python_question(
    question: str,
):
    """Ask a coding related question"""
    try:
        
        assistant = get_python_assistant()
        # user_id = f"user_{user_info['user_id']}"
        user_id = f"user_1"

        
        response = assistant.ask_question(
            question=question,
            user_id=user_id
        )
        
        return {
            "success": True,
            f"\nuser_id": user_id,
            f"\nquestion": question,
            f"\nanswer": response
        }
        
    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=f"Error processing question: {str(e)}"
        )


In [17]:
from langchain_core.messages import ToolMessage, HumanMessage

def ask_with_trace(question, user_id="eval"):
    
    agent = assistant.builder.compile(checkpointer=assistant.checkpointer)
    
    # print(f"Agent input schema: {agent.input_schema.schema()}")
    
    result = agent.invoke(
        {"messages": [HumanMessage(content=question)]},
        {"configurable": {"thread_id": user_id}}
    )

    messages = result["messages"]

    used_tool = any(isinstance(m, ToolMessage) for m in messages)

    return {
        "question": question,
        "used_retrieval": used_tool,
        "final_answer": messages[-1].content
    }


#### Testing Queries

In [18]:
faq = [
    "What is Python?",
    "Explain variables in Python",
    "What is a function?",
    "What does list comprehension mean?",
    "What is object-oriented programming?"
]

general_results = [ask_with_trace(q, "general") for q in faq]

for r in general_results:
    print("\nQ:", r["question"])
    print("Retrieval used:", r["used_retrieval"])



Q: What is Python?
Retrieval used: True

Q: Explain variables in Python
Retrieval used: True

Q: What is a function?
Retrieval used: True

Q: What does list comprehension mean?
Retrieval used: True

Q: What is object-oriented programming?
Retrieval used: True


In [ ]:
retrieval_queries = [
    "How do I define request models in FastAPI?",
    "What does PEP 8 say about line length?",
    "How does async work in FastAPI routes?",
    "How do I use dependency injection in FastAPI?",
    "How is asyncio different from threading in Python?"
]

retrieval_results = [ask_with_trace(q, "retrieval") for q in retrieval_queries]

for r in retrieval_results:
    print("\nQ:", r["question"])
    print("Retrieval used:", r["used_retrieval"])


#### Agent Evaluation

In [ ]:
import pandas as pd

evaluation = []

for r in general_results + retrieval_results:
    evaluation.append({
        "Question": r["question"],
        "Retrieval Triggered": r["used_retrieval"],
        "Answer Length": len(r["final_answer"])
    })

df = pd.DataFrame(evaluation)
df


,Question,Retrieval Triggered,Answer Length
0,What is Python?,True,1160
1,Explain variables in Python,True,1836
2,What is a function?,True,1961
3,What does list comprehension mean?,True,1665
4,What is object-oriented programming?,True,2476
5,How do I define request models in FastAPI?,True,1987
6,What does PEP 8 say about line length?,True,487
7,How does async work in FastAPI routes?,True,2086
8,How do I use dependency injection in FastAPI?,True,2807
9,How is asyncio different from threading in Pyt...,True,2899
